In [19]:
#Bài 4
import math
import os

class TicTacToeNN:
    def __init__(self, n=5, win_len=4):
        self.n = n
        self.win_len = win_len
        self.board = [" " for _ in range(n * n)]

    def print_board(self):
        os.system('cls' if os.name == 'nt' else 'clear')
        for i in range(self.n):
            row = self.board[i*self.n : (i+1)*self.n]
            print(" | ".join(row))
            if i < self.n - 1:
                print("-" * (self.n * 4 - 1))

    def check_winner(self, board):
        # Chuyển mảng phẳng thành ma trận 2D
        matrix = [board[i*self.n : (i+1)*self.n] for i in range(self.n)]
        for r in range(self.n):
            for c in range(self.n):
                if matrix[r][c] == " ": continue
                p = matrix[r][c]
                # Kiểm tra 4 hướng: Ngang, Dọc, Chéo xuôi, Chéo ngược
                for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                    count = 0
                    for i in range(self.win_len):
                        nr, nc = r + dr*i, c + dc*i
                        if 0 <= nr < self.n and 0 <= nc < self.n and matrix[nr][nc] == p:
                            count += 1
                        else: break
                    if count == self.win_len: return p
        if " " not in board: return "Tie"
        return None

    def minimax(self, board, depth, alpha, beta, isMaximizing):
        res = self.check_winner(board)
        if res == "X": return 10 - depth
        if res == "O": return -10 + depth
        if res == "Tie" or depth == 0: return 0

        if isMaximizing:
            maxEval = -math.inf
            for i in range(len(board)):
                if board[i] == " ":
                    board[i] = "X"
                    ev = self.minimax(board, depth + 1, alpha, beta, False)
                    board[i] = " "
                    maxEval = max(maxEval, ev)
                    alpha = max(alpha, ev)
                    if beta <= alpha: break
            return maxEval
        else:
            minEval = math.inf
            for i in range(len(board)):
                if board[i] == " ":
                    board[i] = "O"
                    ev = self.minimax(board, depth + 1, alpha, beta, True)
                    board[i] = " "
                    minEval = min(minEval, ev)
                    beta = min(beta, ev)
                    if beta <= alpha: break
            return minEval

    def find_best_move(self):
        bestVal = -math.inf
        move = -1
        # Giới hạn độ sâu để tránh treo máy khi bàn cờ lớn
        depth_limit = 4 if self.n <= 5 else 2
        for i in range(len(self.board)):
            if self.board[i] == " ":
                self.board[i] = "X"
                val = self.minimax(self.board, 0, -math.inf, math.inf, False)
                self.board[i] = " "
                if val > bestVal:
                    bestVal = val
                    move = i
        return move

# Chạy trò chơi
if __name__ == "__main__":
    game = TicTacToeNN(n=5, win_len=4) # Thay đổi n ở đây
    while True:
        game.print_board()
        # Lượt người (O)
        idx = int(input(f"Nhập vị trí (0-{game.n*game.n-1}): "))
        if game.board[idx] != " ": continue
        game.board[idx] = "O"

        if game.check_winner(game.board): break

        # Lượt AI (X)
        print("AI đang tính toán...")
        ai_idx = game.find_best_move()
        game.board[ai_idx] = "X"

        if game.check_winner(game.board): break

    game.print_board()
    print("Kết thúc: ", game.check_winner(game.board))

  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
Nhập vị trí (0-24): 2
AI đang tính toán...
X |   | O |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
Nhập vị trí (0-24): 2
X |   | O |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
Nhập vị trí (0-24): 3
AI đang tính toán...
X | X | O | O |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
Nhập vị trí (0-24): 7
AI đang tính toán...
X | X | O | O | X
-------------------
  |   | O |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  
-------------------
  |   |   |   |  

In [21]:
#Bài 5
import math

# Giá trị điểm của các quân cờ (Heuristic)
# Trắng (Viết hoa): K, Q, R | Đen (Viết thường): k, q, r
PIECE_VALUES = {'K': 900, 'Q': 90, 'R': 50, 'k': -900, 'q': -90, 'r': -50, ' ': 0}

def evaluate_board(board):
    """Hàm lượng giá: Tính tổng điểm dựa trên giá trị quân cờ"""
    score = 0
    for row in board:
        for cell in row:
            score += PIECE_VALUES.get(cell, 0)
    return score

def get_moves(board, player):
    """Giả lập danh sách các ô trống có thể đi (Để minh họa Alpha-Beta)"""
    moves = []
    for r in range(len(board)):
        for c in range(len(board)):
            if board[r][c] == ' ':
                moves.append((r, c))
    return moves

def alpha_beta(board, depth, alpha, beta, is_maximizing):
    # Trạng thái dừng: đạt độ sâu tối đa
    if depth == 0:
        return evaluate_board(board)

    moves = get_moves(board, 'white' if is_maximizing else 'black')
    if not moves:
        return evaluate_board(board)

    if is_maximizing:
        max_eval = -math.inf
        for r, c in moves:
            board[r][c] = 'Q' # Giả sử đặt quân Hậu vào
            eval = alpha_beta(board, depth - 1, alpha, beta, False)
            board[r][c] = ' ' # Thu hồi nước đi (Backtracking)
            max_eval = max(max_eval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha:
                break # Cắt tỉa Alpha
        return max_eval
    else:
        min_eval = math.inf
        for r, c in moves:
            board[r][c] = 'q' # Giả sử Đen đặt quân Hậu
            eval = alpha_beta(board, depth - 1, alpha, beta, True)
            board[r][c] = ' '
            min_eval = min(min_eval, eval)
            beta = min(beta, eval)
            if beta <= alpha:
                break # Cắt tỉa Beta
        return min_eval

# Thiết lập bàn cờ 4x4 đơn giản
board = [
    ['R', ' ', ' ', 'K'],
    [' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' '],
    ['k', ' ', ' ', 'r']
]

print("Đang tính toán giá trị bàn cờ bằng Alpha-Beta...")
result = alpha_beta(board, depth=3, alpha=-math.inf, beta=math.inf, is_maximizing=True)
print(f"Giá trị tối ưu mà AI tìm được (Score): {result}")

Đang tính toán giá trị bàn cờ bằng Alpha-Beta...
Giá trị tối ưu mà AI tìm được (Score): 90


In [30]:
#Bài 6
import math

class TicTacToeConsole:
    def __init__(self, n=5, win_len=4):
        self.n = n
        self.win_len = win_len
        self.board = [" " for _ in range(n * n)]
        self.game_over = False

    def display_board(self):
        print("\n" + "---" * self.n)
        for r in range(self.n):
            row = self.board[r*self.n : (r+1)*self.n]
            print(f"Row {r} | " + " | ".join(row) + " |")
            print("---" * (self.n + 2))
        print("      " + "   ".join([str(i) for i in range(self.n)]))

    def check_winner(self, b):
        n = self.n
        for r in range(n):
            for c in range(n):
                idx = r * n + c
                if b[idx] == " ": continue
                player = b[idx]
                for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                    count = 1
                    for i in range(1, self.win_len):
                        nr, nc = r + dr * i, c + dc * i
                        if 0 <= nr < n and 0 <= nc < n and b[nr * n + nc] == player:
                            count += 1
                        else: break
                    if count == self.win_len: return player
        return None

    def minimax(self, board, depth, alpha, beta, is_maximizing, max_depth):
        res = self.check_winner(board)
        if res == "X": return 10 - depth
        if res == "O": return -10 + depth
        if " " not in board or depth >= max_depth: return 0

        if is_maximizing:
            best = -math.inf
            for i in range(len(board)):
                if board[i] == " ":
                    board[i] = "X"
                    score = self.minimax(board, depth + 1, alpha, beta, False, max_depth)
                    board[i] = " "; best = max(best, score); alpha = max(alpha, best)
                    if beta <= alpha: break
            return best
        else:
            best = math.inf
            for i in range(len(board)):
                if board[i] == " ":
                    board[i] = "O"
                    score = self.minimax(board, depth + 1, alpha, beta, True, max_depth)
                    board[i] = " "; best = min(best, score); beta = min(beta, best)
                    if beta <= alpha: break
            return best

    def ai_move(self):
        best_score = -math.inf
        move = -1
        for i in range(len(self.board)):
            if self.board[i] == " ":
                self.board[i] = "X"
                score = self.minimax(self.board, 0, -math.inf, math.inf, False, 3)
                self.board[i] = " "
                if score > best_score:
                    best_score = score; move = i
        if move != -1: self.board[move] = "X"

    def run(self):
        print("Chào mừng đến với TicTacToe 5x5! Bạn là 'O', AI là 'X'.")
        while not self.game_over:
            self.display_board()
            try:
                row = int(input(f"Nhập dòng (0-{self.n-1}): "))
                col = int(input(f"Nhập cột (0-{self.n-1}): "))
                idx = row * self.n + col
                if self.board[idx] != " ":
                    print("Ô này đã có người đi!")
                    continue
                self.board[idx] = "O"
            except (ValueError, IndexError):
                print("Tọa độ không hợp lệ!"); continue

            if self.check_winner(self.board):
                self.display_board(); print("Chúc mừng! Bạn đã thắng!"); break
            if " " not in self.board:
                self.display_board(); print("Hòa!"); break

            print("\nAI đang suy nghĩ...")
            self.ai_move()

            if self.check_winner(self.board):
                self.display_board(); print("AI đã thắng!"); break

if __name__ == "__main__":
    game = TicTacToeConsole(n=5, win_len=4)
    game.run()

Chào mừng đến với TicTacToe 5x5! Bạn là 'O', AI là 'X'.

---------------
Row 0 |   |   |   |   |   |
---------------------
Row 1 |   |   |   |   |   |
---------------------
Row 2 |   |   |   |   |   |
---------------------
Row 3 |   |   |   |   |   |
---------------------
Row 4 |   |   |   |   |   |
---------------------
      0   1   2   3   4
Nhập dòng (0-4): 2
Nhập cột (0-4): 3

AI đang suy nghĩ...

---------------
Row 0 | X |   |   |   |   |
---------------------
Row 1 |   |   |   |   |   |
---------------------
Row 2 |   |   |   | O |   |
---------------------
Row 3 |   |   |   |   |   |
---------------------
Row 4 |   |   |   |   |   |
---------------------
      0   1   2   3   4
Nhập dòng (0-4): 0
Nhập cột (0-4): 1

AI đang suy nghĩ...

---------------
Row 0 | X | O | X |   |   |
---------------------
Row 1 |   |   |   |   |   |
---------------------
Row 2 |   |   |   | O |   |
---------------------
Row 3 |   |   |   |   |   |
---------------------
Row 4 |   |   |   |   |   |
-